# Population Gaze Position — Box & Whisker Plots

For each activity, plots the distribution of raw yaw and pitch gaze positions across the population, grouped by **age group** and **gender**.

Each box pools all gaze samples from every participant in the group — the 1D equivalent of the density heatmap.

In [ ]:
from pathlib import Path
import nymeria_gaze_tools as ngt

DATA_ROOT = Path("../data/processed")

In [ ]:
catalog = ngt.load_metadata(data_root=DATA_ROOT)

activities = ngt.list_activities(catalog)
age_groups = sorted(catalog["participant_age_group"].dropna().unique())
genders    = sorted(catalog["participant_gender"].dropna().unique())

print(f"{len(catalog)} sessions | {len(activities)} activities")
print("Age groups:", age_groups)
print("Genders:   ", genders)

In [ ]:
# Load and preprocess every session once — reused across all plot loops
all_sessions = ngt.filter_sessions(catalog, has_gaze_data=True)
session_cache = {}

for i, uid in enumerate(all_sessions["sequence_uid"], start=1):
    print(f"[{i}/{len(all_sessions)}] {uid}")
    try:
        raw = ngt.load_session(uid, data_root=DATA_ROOT)
        session_cache[uid] = ngt.preprocess(raw)
    except Exception as e:
        print(f"  Skipping {uid}: {e}")

print(f"\nCached {len(session_cache)} sessions")

---
## 1 · By age group — yaw and pitch per activity

In [ ]:
for activity in activities:
    print(f"\n{'='*60}\nActivity: {activity}\n{'='*60}")

    groups = {}
    for age in age_groups:
        sessions = ngt.filter_sessions(
            catalog, script=activity, participant_age_group=age, has_gaze_data=True,
        )
        dfs = [session_cache[uid] for uid in sessions["sequence_uid"] if uid in session_cache]
        if not dfs:
            print(f"  [{age}] — no sessions, skipping")
            continue
        print(f"  [{age}] — {len(dfs)} sessions")
        groups[age] = dfs

    if not groups:
        continue

    ngt.plot_gaze_position_boxplots(
        groups, column="avg_yaw_deg",
        title=f"{activity}  |  Yaw by age group",
    ).show()

    ngt.plot_gaze_position_boxplots(
        groups, column="pitch_deg",
        title=f"{activity}  |  Pitch by age group",
    ).show()

---
## 2 · By gender — yaw and pitch per activity

In [ ]:
for activity in activities:
    print(f"\n{'='*60}\nActivity: {activity}\n{'='*60}")

    groups = {}
    for gender in genders:
        sessions = ngt.filter_sessions(
            catalog, script=activity, participant_gender=gender, has_gaze_data=True,
        )
        dfs = [session_cache[uid] for uid in sessions["sequence_uid"] if uid in session_cache]
        if not dfs:
            print(f"  [{gender}] — no sessions, skipping")
            continue
        print(f"  [{gender}] — {len(dfs)} sessions")
        groups[gender] = dfs

    if not groups:
        continue

    ngt.plot_gaze_position_boxplots(
        groups, column="avg_yaw_deg",
        title=f"{activity}  |  Yaw by gender",
    ).show()

    ngt.plot_gaze_position_boxplots(
        groups, column="pitch_deg",
        title=f"{activity}  |  Pitch by gender",
    ).show()